# Importação dos dados

In [1]:
import pandas as pd

DATA_PATH = '../data/'
file_name = 'hour.csv'

df = pd.read_csv(DATA_PATH+file_name, sep=',')

df.shape

(17379, 17)

In [2]:
df.head(1)

,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16


# Verificação dos tipos de dados

In [3]:
df.dtypes

instant         int64
dteday         object
season          int64
yr              int64
mnth            int64
hr              int64
holiday         int64
weekday         int64
workingday      int64
weathersit      int64
temp          float64
atemp         float64
hum           float64
windspeed     float64
casual          int64
registered      int64
cnt             int64
dtype: object

# Verifica valores duplicados

In [4]:
# verifica se existe valores duplicados

df.duplicated().sum()

np.int64(0)

In [5]:
for column in df.columns:
    # ^ inicio da string
    # $ fim da string
    # \s identifica espaços
    # * especifica que o replace será aplicado independente da quantidade de espaços
    print(f'{column}: {df[column].replace(r'^\s*$', pd.NA, regex=True).isna().sum()}')

instant: 0
dteday: 0
season: 0
yr: 0
mnth: 0
hr: 0
holiday: 0
weekday: 0
workingday: 0
weathersit: 0
temp: 0
atemp: 0
hum: 0
windspeed: 0
casual: 0
registered: 0
cnt: 0


# Verifica os valores unicos de cada coluna

In [6]:
# Verifica se os valores fazem sentido para a coluna
for column in df.drop(columns=['instant', 'dteday']).columns:
    print(f'{column}: {df[column].unique()}')

season: [1 2 3 4]
yr: [0 1]
mnth: [ 1  2  3  4  5  6  7  8  9 10 11 12]
hr: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23]
holiday: [0 1]
weekday: [6 0 1 2 3 4 5]
workingday: [0 1]
weathersit: [1 2 3 4]
temp: [0.24 0.22 0.2  0.32 0.38 0.36 0.42 0.46 0.44 0.4  0.34 0.3  0.26 0.16
 0.14 0.18 0.12 0.28 0.1  0.08 0.06 0.04 0.02 0.52 0.56 0.58 0.6  0.48
 0.54 0.5  0.66 0.64 0.62 0.68 0.7  0.74 0.76 0.72 0.78 0.82 0.8  0.86
 0.88 0.9  0.84 0.92 0.94 0.96 0.98 1.  ]
atemp: [0.2879 0.2727 0.2576 0.3485 0.3939 0.3333 0.4242 0.4545 0.4394 0.4091
 0.2273 0.2121 0.197  0.1667 0.1364 0.1061 0.1212 0.1818 0.2424 0.1515
 0.3182 0.0606 0.0758 0.0909 0.303  0.0303 0.0455 0.     0.0152 0.3636
 0.5    0.5303 0.5455 0.5909 0.4697 0.5152 0.6212 0.6061 0.4848 0.3788
 0.6364 0.6515 0.6667 0.5758 0.5606 0.6818 0.697  0.7424 0.7727 0.7576
 0.7273 0.7121 0.803  0.7879 0.8333 0.8182 0.8485 0.8788 0.8636 0.8939
 0.9242 0.9091 0.9545 0.9848 1.    ]
hum: [0.81 0.8  0.75 0.86 0.76 0.77 0.7

# Verifica período de cobertura do dataset

In [7]:
print("Primeira data:", df["dteday"].min())
print("Última data:", df["dteday"].max())

Primeira data: 2011-01-01
Última data: 2012-12-31


## Criação do datetime para melhor representação do momento do registro

In [8]:
df["datetime"] = pd.to_datetime(df["dteday"]) + pd.to_timedelta(df["hr"], unit="h")
df["datetime"]

0       2011-01-01 00:00:00
1       2011-01-01 01:00:00
2       2011-01-01 02:00:00
3       2011-01-01 03:00:00
4       2011-01-01 04:00:00
                ...        
17374   2012-12-31 19:00:00
17375   2012-12-31 20:00:00
17376   2012-12-31 21:00:00
17377   2012-12-31 22:00:00
17378   2012-12-31 23:00:00
Name: datetime, Length: 17379, dtype: datetime64[ns]

## Verifica ordenação das datas

In [9]:
# True = ordenada
# False = desordenada
df['datetime'].is_monotonic_increasing

True

## Verifica duplicidade de datas

In [10]:
df['datetime'].duplicated().sum()

np.int64(0)

# Verifica horas faltantes

In [11]:
# Define o período esperado
expected_range = pd.date_range(start=df['datetime'].min(), end=df['datetime'].max(), freq='h')

# verifica quais valores existem em expected_range mas não existem no datetime
missing_hours = expected_range.difference(df['datetime'])

len(missing_hours)

165

In [12]:
missing_hours

DatetimeIndex(['2011-01-02 05:00:00', '2011-01-03 02:00:00',
               '2011-01-03 03:00:00', '2011-01-04 03:00:00',
               '2011-01-05 03:00:00', '2011-01-06 03:00:00',
               '2011-01-07 03:00:00', '2011-01-11 03:00:00',
               '2011-01-11 04:00:00', '2011-01-12 03:00:00',
               ...
               '2012-10-30 07:00:00', '2012-10-30 08:00:00',
               '2012-10-30 09:00:00', '2012-10-30 10:00:00',
               '2012-10-30 11:00:00', '2012-10-30 12:00:00',
               '2012-11-08 03:00:00', '2012-11-29 03:00:00',
               '2012-12-24 04:00:00', '2012-12-25 03:00:00'],
              dtype='datetime64[ns]', length=165, freq=None)

## Identifica blocos de horas faltantes

Durante a verificação da estrutura temporal, foram identificados 75 blocos de horários faltantes, totalizando 165 horas sem registros.

Para identificar essas lacunas, os timestamps existentes foram comparados com uma sequência horária contínua utilizando pd.date_range() e difference(). Os horários ausentes foram posteriormente agrupados para identificar os períodos consecutivos sem registros.

Essa etapa é importante para avaliar a continuidade da série temporal antes de iniciar a EDA

In [13]:
# Cria um dataframe com as horas faltantes
missing_df = pd.DataFrame({
    'datetime': missing_hours
})

missing_df.head()

,datetime
0,2011-01-02 05:00:00
1,2011-01-03 02:00:00
2,2011-01-03 03:00:00
3,2011-01-04 03:00:00
4,2011-01-05 03:00:00


In [14]:
# Calcula a diferença com relação ao valor anterior ao atual
missing_df['time_diff'] = missing_df['datetime'].diff()

missing_df['group'] = (
    
    # verifica se a diferença do horários atual é superior a 1 hora em relação ao horário anterior
    missing_df['time_diff'] != pd.Timedelta(hours=1)
    
).cumsum() # cria a soma acumulada
missing_df

,datetime,time_diff,group
0,2011-01-02 05:00:00,NaT,1
1,2011-01-03 02:00:00,0 days 21:00:00,2
2,2011-01-03 03:00:00,0 days 01:00:00,2
3,2011-01-04 03:00:00,1 days 00:00:00,3
4,2011-01-05 03:00:00,1 days 00:00:00,4
...,...,...,...
160,2012-10-30 12:00:00,0 days 01:00:00,71
161,2012-11-08 03:00:00,8 days 15:00:00,72
162,2012-11-29 03:00:00,21 days 00:00:00,73
163,2012-12-24 04:00:00,25 days 01:00:00,74


In [15]:
missing_periods = (
    missing_df
    .groupby('group') # resume os grupos criados de acordo com group em missing_df
    .agg(
        start_datetime=('datetime', 'min'), # inicio da período
        end_datetime=('datetime', 'max'), # fim do período
        missing_hours=('datetime', 'count') # quantidade de horas faltantes do período
    )
    .reset_index(drop=True)
)

missing_periods

,start_datetime,end_datetime,missing_hours
0,2011-01-02 05:00:00,2011-01-02 05:00:00,1
1,2011-01-03 02:00:00,2011-01-03 03:00:00,2
2,2011-01-04 03:00:00,2011-01-04 03:00:00,1
3,2011-01-05 03:00:00,2011-01-05 03:00:00,1
4,2011-01-06 03:00:00,2011-01-06 03:00:00,1
...,...,...,...
70,2012-10-29 01:00:00,2012-10-30 12:00:00,36
71,2012-11-08 03:00:00,2012-11-08 03:00:00,1
72,2012-11-29 03:00:00,2012-11-29 03:00:00,1
73,2012-12-24 04:00:00,2012-12-24 04:00:00,1


In [16]:
print('Horas faltantes: ', missing_periods['missing_hours'].sum())

Horas faltantes:  165


# Renomeando colunas

In [17]:
# Renomeia as colunas
df = df.rename(columns={
    'dteday':'date',
    'yr':'year',
    'mnth':'month',
    'hr':'hour',
    'workingday':'working_day',
    'weathersit': 'weather_sit',
    'atemp':'feeling_temp',
    'windspeed': 'wind_speed'
})

# Remove a coluna date

In [18]:
df = df.drop(columns='date', axis=1)

# Reposiciona coluna cnt para o final do dataset

In [19]:
cnt = df.pop('cnt')
df.insert(len(df.columns), 'cnt', cnt)
df.head()

,instant,season,year,month,hour,holiday,weekday,working_day,weather_sit,temp,feeling_temp,hum,wind_speed,casual,registered,datetime,cnt
0,1,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,2011-01-01 00:00:00,16
1,2,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,2011-01-01 01:00:00,40
2,3,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,2011-01-01 02:00:00,32
3,4,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,2011-01-01 03:00:00,13
4,5,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,2011-01-01 04:00:00,1


# Salva o dataset

In [20]:
file_name = 'bike_sharing_silver.csv'
df.to_csv(DATA_PATH+file_name, sep=',', index=None)